# LeetCode 3526 — Range XOR Queries with Subarray Reversals bằng Python

- Link: https://leetcode.com/problems/range-xor-queries-with-subarray-reversals/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: **Treap ẩn chỉ số** (implicit treap / FHQ treap) với **lazy đảo đoạn**, tổng hợp **XOR cây con**, chia căn (sqrt decomposition)

> **Ghi chú Premium:** đề bài viết lại theo bản mô tả công khai trên doocs (leetcode.doocs.org) vì nội dung gốc chỉ dành cho tài khoản Premium. Trang doocs **không có code Python** cho bài này, nên **tên hàm `getResults` và chữ ký là suy luận** (theo quy ước đặt tên thường gặp của LeetCode) — cần đối chiếu skeleton thật nếu có tài khoản Premium.

Skeleton (suy luận):

```python
class Solution:
    def getResults(self, nums: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra ba thao tác cần hỗ trợ cùng lúc: **gán một phần tử**, **XOR một đoạn**, **đảo ngược một đoạn** — mảng thường hay cây phân đoạn thường không làm được thao tác đảo đoạn nhanh.
2. Hiểu **treap ẩn chỉ số**: cây nhị phân mà thứ tự trung tự (in-order) chính là mảng; mỗi nút lưu kích thước cây con để tìm "phần tử thứ k".
3. Cài **split / merge** và cờ **lazy reverse** (đổi chỗ con trái/phải, đẩy cờ xuống khi cần).
4. Tận dụng tính chất: **XOR không phụ thuộc thứ tự** ⇒ đảo đoạn không đổi XOR của đoạn đó, nên cờ đảo không cần cập nhật giá trị tổng hợp.
5. So sánh với chia căn (sqrt decomposition) và vét cạn.

## 2. Đề bài & ví dụ

Cho mảng `nums` độ dài `n` và `q` truy vấn, mỗi truy vấn là một trong ba loại:

1. `[1, index, value]`: gán `nums[index] = value`.
2. `[2, left, right]`: tính XOR của `nums[left..right]` và **ghi lại** kết quả.
3. `[3, left, right]`: **đảo ngược** đoạn `nums[left..right]` tại chỗ.

Trả về danh sách kết quả các truy vấn loại 2 theo thứ tự.

**Ví dụ 1:** `nums = [1,2,3,4,5]`, `queries = [[2,1,3],[1,2,10],[3,0,4],[2,0,4]]` → `[5,8]`.

**Ví dụ 2:** `nums = [7,8,9]`, `queries = [[1,0,3],[2,0,2],[3,1,2]]` → `[2]`.

**Ràng buộc (theo doocs):** `1 ≤ n, q ≤ 10^5`, `0 ≤ nums[i], value ≤ 10^9`, `0 ≤ left ≤ right < n`.

In [1]:
from typing import List
import random, sys
from functools import reduce
from operator import xor
sys.setrecursionlimit(10000)
examples = [
    ([1, 2, 3, 4, 5], [[2, 1, 3], [1, 2, 10], [3, 0, 4], [2, 0, 4]], [5, 8]),
    ([7, 8, 9], [[1, 0, 3], [2, 0, 2], [3, 1, 2]], [2]),
]
for nums, qs, exp in examples:
    print(nums, qs, '-> kỳ vọng', exp)

[1, 2, 3, 4, 5] [[2, 1, 3], [1, 2, 10], [3, 0, 4], [2, 0, 4]] -> kỳ vọng [5, 8]
[7, 8, 9] [[1, 0, 3], [2, 0, 2], [3, 1, 2]] -> kỳ vọng [2]


## 3. Kiến thức nền

### 3.1. XOR và tính giao hoán

`a ^ b ^ c` không phụ thuộc thứ tự ⇒ đảo một đoạn không đổi XOR của **chính đoạn đó**; nhưng đoạn truy vấn sau có thể **cắt ngang** đoạn đã đảo nên vẫn phải theo dõi vị trí.

In [2]:
a = [5, 4, 10, 2, 1]
print(reduce(xor, a), reduce(xor, a[::-1]), '| XOR đoạn [1,2] trước/sau đảo cả mảng:', a[1] ^ a[2], a[::-1][1] ^ a[::-1][2])

8 8 | XOR đoạn [1,2] trước/sau đảo cả mảng: 14 8


### 3.2. Cây nhị phân "ẩn chỉ số"

Không lưu chỉ số trong nút. Vị trí của nút = (kích thước cây con trái) + (số nút đứng trước nó ở các tổ tiên). Mỗi nút lưu `size` để biết cây con trái có bao nhiêu phần tử.

### 3.3. Treap: cây + heap theo độ ưu tiên ngẫu nhiên

Mỗi nút có `priority` ngẫu nhiên; cây là heap theo priority (cha ≥ con) ⇒ độ sâu kỳ vọng `O(log n)`.

- `split(t, k)`: tách cây `t` thành (k phần tử đầu, phần còn lại).
- `merge(a, b)`: ghép hai cây (mọi phần tử của `a` đứng trước `b`), giữ tính heap.

In [3]:
class Node:
    __slots__ = ('val', 'agg', 'size', 'pri', 'left', 'right', 'rev')
    def __init__(self, val, pri):
        self.val = self.agg = val
        self.size, self.pri = 1, pri
        self.left = self.right = None
        self.rev = False

def size(t): return t.size if t else 0
def agg(t): return t.agg if t else 0

def pull(t):
    t.size = 1 + size(t.left) + size(t.right)
    t.agg = t.val ^ agg(t.left) ^ agg(t.right)

def push(t):
    if t.rev:
        t.left, t.right = t.right, t.left
        if t.left: t.left.rev = not t.left.rev
        if t.right: t.right.rev = not t.right.rev
        t.rev = False

def split(t, k):
    if t is None:
        return None, None
    push(t)
    if size(t.left) >= k:
        a, b = split(t.left, k)
        t.left = b; pull(t)
        return a, t
    a, b = split(t.right, k - size(t.left) - 1)
    t.right = a; pull(t)
    return t, b

def merge(a, b):
    if a is None or b is None:
        return a or b
    if a.pri > b.pri:
        push(a); a.right = merge(a.right, b); pull(a); return a
    push(b); b.left = merge(a, b.left); pull(b); return b

def to_list(t, out=None):
    out = [] if out is None else out
    if t:
        push(t); to_list(t.left, out); out.append(t.val); to_list(t.right, out)
    return out

rnd = random.Random(1)
root = None
for x in [1, 2, 3, 4, 5]:
    root = merge(root, Node(x, rnd.random()))
a, b = split(root, 2)
print('split 2:', to_list(a), to_list(b))
root = merge(a, b)
print('merge lại:', to_list(root))

split 2: [1, 2] [3, 4, 5]
merge lại: [1, 2, 3, 4, 5]


### 3.4. Lazy reverse

Đảo đoạn = tách ra cây con chứa đúng đoạn đó và **bật cờ `rev`** ở gốc của nó. Khi nào cần đi vào cây con, `push` sẽ đổi chỗ trái/phải và đẩy cờ xuống. Giá trị `agg` (XOR) không đổi khi đảo.

In [4]:
a, rest = split(root, 1)
m, c = split(rest, 3)          # m = phần tử chỉ số 1..3
m.rev = not m.rev
root = merge(a, merge(m, c))
print('đảo [1,3]:', to_list(root))

đảo [1,3]: [1, 4, 3, 2, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1:

| bước | truy vấn | mảng | kết quả |
|---|---|---|---|
| 1 | XOR [1,3] | `[1,2,3,4,5]` | `2^3^4 = 5` |
| 2 | gán `nums[2] = 10` | `[1,2,10,4,5]` | |
| 3 | đảo [0,4] | `[5,4,10,2,1]` | |
| 4 | XOR [0,4] | | `5^4^10^2^1 = 8` |

Nút thắt khi làm trực tiếp: đảo đoạn dài và XOR đoạn dài đều `O(n)` mỗi lần ⇒ `O(nq) = 10^10`. Cần cấu trúc mà **cắt-ghép đoạn** rẻ: treap ẩn chỉ số (`O(log n)` kỳ vọng mỗi thao tác) hoặc chia khối (`O(√n)`).

## 5. Cách 1 — Brute Force: mô phỏng trên `list`

In [5]:
def range_xor_brute(nums, queries):
    a = nums[:]
    res = []
    for t, x, y in queries:
        if t == 1:
            a[x] = y
        elif t == 2:
            res.append(reduce(xor, a[x:y + 1], 0))
        else:
            a[x:y + 1] = a[x:y + 1][::-1]
    return res

for nums, qs, exp in examples:
    print(range_xor_brute(nums, qs), '| kỳ vọng', exp)

[5, 8] | kỳ vọng [5, 8]
[2] | kỳ vọng [2]


**Độ phức tạp:** `O(n)` mỗi truy vấn loại 2/3 ⇒ `O(n·q)` xấu nhất.

**Nút thắt:** cả XOR lẫn đảo đều phải "chạm" từng phần tử của đoạn.

## 6. Ý tưởng thuật toán / bước đột phá

**Treap ẩn chỉ số** biểu diễn mảng; mỗi nút lưu `val`, `agg` (XOR cây con), `size`, cờ `rev`.

- **Gán** `nums[i] = v`: `split(root, i) → (A, B)`, `split(B, 1) → (M, C)`; đặt `M.val = M.agg = v`; ghép lại.
- **XOR** `[l, r]`: `split(root, l) → (A, B)`, `split(B, r − l + 1) → (M, C)`; đáp án `M.agg`; ghép lại.
- **Đảo** `[l, r]`: như trên, bật `M.rev`; ghép lại.

Mỗi thao tác `O(log n)` kỳ vọng. Điểm mấu chốt: **cờ đảo không làm thay đổi `agg`** vì XOR giao hoán — nếu bài hỏi tổng có hướng (ví dụ băm chuỗi) thì phải lưu thêm giá trị theo chiều ngược.

**Chia khối (cách 2):** chia mảng thành các khối, mỗi khối lưu danh sách, XOR, cờ đảo. Đảo đoạn = tách khối ở hai biên rồi đảo **thứ tự các khối** và bật cờ từng khối; XOR đoạn = tách biên rồi XOR các khối. Định kỳ xây lại để số khối không tăng mãi.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chia khối (sqrt decomposition) với cờ đảo

In [6]:
class BlockList:
    def __init__(self, nums, B=None):
        self.B = B or max(16, int(len(nums) ** 0.5) * 2)
        self._build(nums)
        self.limit = len(self.blocks) + 1          # số khối ban đầu; vượt 3 lần thì xây lại

    def _build(self, arr):
        B = self.B
        self.blocks = [arr[i:i + B] for i in range(0, len(arr), B)]
        self.xs = [reduce(xor, b, 0) for b in self.blocks]
        self.rev = [False] * len(self.blocks)

    def _flatten(self):
        out = []
        for b, r in zip(self.blocks, self.rev):
            out.extend(b[::-1] if r else b)
        return out

    def _materialize(self, i):
        if self.rev[i]:
            self.blocks[i].reverse()
            self.rev[i] = False

    def _split(self, pos):
        """Bảo đảm có ranh giới khối tại vị trí pos; trả về chỉ số khối bắt đầu tại pos."""
        acc = 0
        for i, b in enumerate(self.blocks):
            if acc == pos:
                return i
            if acc + len(b) > pos:
                self._materialize(i)
                off = pos - acc
                left, right = b[:off], b[off:]
                xl = reduce(xor, left, 0)
                self.blocks[i:i + 1] = [left, right]
                self.xs[i:i + 1] = [xl, self.xs[i] ^ xl]
                self.rev[i:i + 1] = [False, False]
                return i + 1
            acc += len(b)
        return len(self.blocks)

    def update(self, idx, val):
        acc = 0
        for i, b in enumerate(self.blocks):
            if acc + len(b) > idx:
                self._materialize(i)
                old = b[idx - acc]
                b[idx - acc] = val
                self.xs[i] ^= old ^ val
                return
            acc += len(b)

    def range_xor(self, l, r):
        i = self._split(l)
        j = self._split(r + 1)
        res = reduce(xor, self.xs[i:j], 0)
        self._check()
        return res

    def reverse(self, l, r):
        i = self._split(l)
        j = self._split(r + 1)
        self.blocks[i:j] = self.blocks[i:j][::-1]
        self.xs[i:j] = self.xs[i:j][::-1]
        self.rev[i:j] = [not f for f in self.rev[i:j][::-1]]
        self._check()

    def _check(self):
        if len(self.blocks) > 3 * self.limit:
            self._build(self._flatten())

def range_xor_blocks(nums, queries):
    bl = BlockList(nums)
    res = []
    for t, x, y in queries:
        if t == 1:
            bl.update(x, y)
        elif t == 2:
            res.append(bl.range_xor(x, y))
        else:
            bl.reverse(x, y)
    return res

for nums, qs, exp in examples:
    print(range_xor_blocks(nums, qs), '| kỳ vọng', exp)

[5, 8] | kỳ vọng [5, 8]
[2] | kỳ vọng [2]


**Độ phức tạp:** với khối cỡ `B`, mỗi thao tác `O(n/B + B)` (duyệt danh sách khối + tách một khối); xây lại `O(n)` sau mỗi `~n/B` thao tác. Chọn `B ≈ √n` ⇒ `O(√n)` mỗi thao tác. Trong Python, vòng lặp duyệt khối là phần chậm.

### 7.2. Cách 3 — Treap ẩn chỉ số với lớp `Node` (dễ đọc)

In [7]:
def range_xor_treap_obj(nums, queries, seed=2024):
    rnd = random.Random(seed)
    root = None
    for x in nums:                          # dựng bằng merge liên tiếp: O(n log n)
        root = merge(root, Node(x, rnd.random()))
    res = []
    for t, x, y in queries:
        if t == 1:
            a, b = split(root, x)
            m, c = split(b, 1)
            m.val = m.agg = y
            root = merge(a, merge(m, c))
        else:
            a, b = split(root, x)
            m, c = split(b, y - x + 1)
            if t == 2:
                res.append(m.agg)
            else:
                m.rev = not m.rev
            root = merge(a, merge(m, c))
    return res

for nums, qs, exp in examples:
    print(range_xor_treap_obj(nums, qs), '| kỳ vọng', exp)

[5, 8] | kỳ vọng [5, 8]
[2] | kỳ vọng [2]


### 7.3. Cách 4 — Treap trên mảng song song + dựng `O(n)` (lời giải chính)

Để nhanh hơn trong Python: thay đối tượng bằng các **list song song** (`L, R, V, X, S, RV, P`) chỉ số theo nút, và dựng cây ban đầu trong `O(n)` bằng **ngăn xếp** (cây Descartes theo độ ưu tiên) thay vì `n` lần `merge`.

In [8]:
def range_xor_treap(nums, queries, seed=12345):
    n = len(nums)
    rnd = random.Random(seed)
    L = [0] * (n + 1); R = [0] * (n + 1)
    V = [0] + nums[:]; X = [0] + nums[:]
    S = [0] + [1] * n; RV = [0] * (n + 1)
    P = [0.0] + [rnd.random() for _ in range(n)]

    def pull(t):
        l, r = L[t], R[t]
        S[t] = S[l] + S[r] + 1
        X[t] = X[l] ^ X[r] ^ V[t]

    def push(t):
        if RV[t]:
            l, r = L[t], R[t]
            L[t], R[t] = r, l
            if l: RV[l] ^= 1
            if r: RV[r] ^= 1
            RV[t] = 0

    def split(t, k):
        if not t:
            return 0, 0
        push(t)
        if S[L[t]] >= k:
            a, b = split(L[t], k); L[t] = b; pull(t); return a, t
        a, b = split(R[t], k - S[L[t]] - 1); R[t] = a; pull(t); return t, b

    def merge(a, b):
        if not a or not b:
            return a or b
        if P[a] > P[b]:
            push(a); R[a] = merge(R[a], b); pull(a); return a
        push(b); L[b] = merge(a, L[b]); pull(b); return b

    # dựng cây Descartes O(n): duyệt trái -> phải, ngăn xếp giữ "cạnh phải"
    stack = []
    for i in range(1, n + 1):
        last = 0
        while stack and P[stack[-1]] < P[i]:
            last = stack.pop()
        L[i] = last
        if stack:
            R[stack[-1]] = i
        stack.append(i)
    root = stack[0]
    order, st = [], [root]
    while st:                                  # cập nhật size/agg theo hậu thứ tự
        t = st.pop(); order.append(t)
        if L[t]: st.append(L[t])
        if R[t]: st.append(R[t])
    for t in reversed(order):
        pull(t)

    res = []
    for typ, a, b in queries:
        if typ == 1:
            x, y = split(root, a)
            m, z = split(y, 1)
            V[m] = X[m] = b
            root = merge(x, merge(m, z))
        else:
            x, y = split(root, a)
            m, z = split(y, b - a + 1)
            if typ == 2:
                res.append(X[m])
            else:
                RV[m] ^= 1
            root = merge(x, merge(m, z))
    return res

for nums, qs, exp in examples:
    print(range_xor_treap(nums, qs), '| kỳ vọng', exp)

[5, 8] | kỳ vọng [5, 8]
[2] | kỳ vọng [2]


**Độ phức tạp:** dựng `O(n)`, mỗi truy vấn `O(log n)` kỳ vọng (độ sâu treap ngẫu nhiên ~ `2.5 log2 n`). Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace ví dụ 1 bằng treap (cách 3): sau mỗi truy vấn in mảng hiện tại (duyệt trung tự, có đẩy cờ) và kết quả.

In [9]:
def trace(nums, queries, seed=7):
    rnd = random.Random(seed)
    root = None
    for x in nums:
        root = merge(root, Node(x, rnd.random()))
    print('ban đầu:', to_list(root))
    for t, x, y in queries:
        a, b = split(root, x)
        if t == 1:
            m, c = split(b, 1)
            m.val = m.agg = y
            note = f'gán vị trí {x} = {y}'
        else:
            m, c = split(b, y - x + 1)
            seg = to_list(m)
            if t == 2:
                note = f'XOR {seg} = {m.agg}'
            else:
                m.rev = not m.rev
                note = f'đảo đoạn {seg} (bật cờ rev ở gốc cây con, agg vẫn = {m.agg})'
        root = merge(a, merge(m, c))
        print(f"  {[t, x, y]}: {note} -> mảng {to_list(root)}")

trace(*examples[0][:2])

ban đầu: [1, 2, 3, 4, 5]
  [2, 1, 3]: XOR [2, 3, 4] = 5 -> mảng [1, 2, 3, 4, 5]
  [1, 2, 10]: gán vị trí 2 = 10 -> mảng [1, 2, 10, 4, 5]
  [3, 0, 4]: đảo đoạn [1, 2, 10, 4, 5] (bật cờ rev ở gốc cây con, agg vẫn = 8) -> mảng [5, 4, 10, 2, 1]
  [2, 0, 4]: XOR [5, 4, 10, 2, 1] = 8 -> mảng [5, 4, 10, 2, 1]


## 9. Code nộp LeetCode

(Tên hàm `getResults` là **suy luận** — đối chiếu skeleton thật.)

In [10]:
class Solution:
    def getResults(self, nums: List[int], queries: List[List[int]]) -> List[int]:
        return range_xor_treap(nums, queries)

for nums, qs, exp in examples:
    got = Solution().getResults(nums, qs)
    print(got, 'OK' if got == exp else 'SAI')

[5, 8] OK
[2] OK


Khi nộp, chép thân hàm `range_xor_treap` vào trong method (hoặc định nghĩa nó ở mức module như trên — LeetCode chấp nhận hàm phụ ngoài lớp) và nhớ `sys.setrecursionlimit` (độ sâu đệ quy chỉ `O(log n)` kỳ vọng nhưng nên an toàn).

## 10. Vì sao thuật toán đúng

1. **Bất biến biểu diễn:** duyệt trung tự của treap (sau khi áp dụng mọi cờ `rev`) luôn bằng mảng hiện tại. `split(t, k)` trả về đúng `k` phần tử đầu và phần còn lại vì nó đi theo `size` của cây con trái; `merge` giữ thứ tự (mọi phần tử của `a` đứng trước `b`).
2. **Cờ đảo:** đảo thứ tự trung tự của một cây ⇔ đổi chỗ con trái/phải ở **mọi** nút; cờ lazy trì hoãn việc đó, và `push` luôn được gọi trước khi đi xuống/ghép, nên cấu trúc thấy được luôn đúng.
3. **Tổng hợp XOR:** `agg = val ^ agg(trái) ^ agg(phải)` không phụ thuộc thứ tự trái/phải ⇒ cờ đảo không làm sai `agg`; `pull` sau mỗi thay đổi cấu trúc giữ `agg` và `size` đúng.
4. **Truy vấn:** sau hai lần `split`, cây giữa chứa đúng `nums[l..r]`; trả về `agg` của nó.
5. **Chia khối (cách 2):** tương tự với "nút" là các khối: thứ tự khối + cờ đảo mỗi khối mô tả mảng; XOR khối không đổi khi đảo.

## 11. Phân tích Time / Space Complexity

| Cách | Mỗi truy vấn | Dựng | Space |
|---|---|---|---|
| 1. Mô phỏng list | `O(n)` | `O(n)` | `O(n)` |
| 2. Chia khối | `O(√n)` (khấu hao, kể cả xây lại) | `O(n)` | `O(n)` |
| 3. Treap đối tượng | `O(log n)` kỳ vọng | `O(n log n)` | `O(n)` |
| 4. Treap mảng song song | `O(log n)` kỳ vọng | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {
    'blocks': range_xor_blocks,
    'treap_obj': range_xor_treap_obj,
    'treap': range_xor_treap,
    'submit': lambda a, q: Solution().getResults(a, q),
}
edge = [
    ([0], [[2, 0, 0], [1, 0, 7], [3, 0, 0], [2, 0, 0]], [0, 7]),
    ([1, 2], [[3, 0, 1], [1, 0, 5], [2, 0, 0], [2, 1, 1]], [5, 1]),
    ([5, 5, 5, 5], [[2, 0, 3], [2, 1, 2]], [0, 0]),
    ([10**9, 0, 10**9], [[3, 0, 2], [2, 0, 1]], [10**9]),
]
for nums, qs, exp in examples + edge:
    assert range_xor_brute(nums, qs) == exp
    for name, f in impls.items():
        got = f(nums, qs)
        assert got == exp, (name, nums, qs, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng

In [12]:
def rand_queries(n, q, vmax):
    qs = []
    for _ in range(q):
        t = random.randint(1, 3)
        if t == 1:
            qs.append([1, random.randrange(n), random.randint(0, vmax)])
        else:
            l = random.randrange(n)
            qs.append([t, l, random.randrange(l, n)])
    return qs

random.seed(3526)
cnt = 0
for _ in range(400):
    n = random.randint(1, 40)
    nums = [random.randint(0, 31) for _ in range(n)]
    qs = rand_queries(n, random.randint(1, 40), 31)
    exp = range_xor_brute(nums, qs)
    for name, f in impls.items():
        assert f(nums, qs) == exp, (name, nums, qs)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'bộ test ngẫu nhiên.')

Khớp mô phỏng trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn

In [13]:
import time
random.seed(5)
n = q = 10**5
nums = [random.randint(0, 10**9) for _ in range(n)]
qs = rand_queries(n, q, 10**9)
t0 = time.time(); a = Solution().getResults(nums, qs); t1 = time.time() - t0
# kiểm tra độc lập trên 2.000 truy vấn đầu bằng mô phỏng
assert a[:sum(1 for t, _, _ in qs[:2000] if t == 2)] == range_xor_brute(nums, qs[:2000])
print(f"n = q = 10^5: treap mảng song song {t1:.2f}s, {len(a)} kết quả")
n2 = q2 = 2 * 10**4
nums2 = nums[:n2]; qs2 = rand_queries(n2, q2, 10**9)
t0 = time.time(); b1 = range_xor_blocks(nums2, qs2); tb = time.time() - t0
t0 = time.time(); b2 = range_xor_treap(nums2, qs2); tt = time.time() - t0
assert b1 == b2
print(f"n = q = 2·10^4: chia khối {tb:.2f}s | treap {tt:.2f}s")

n = q = 10^5: treap mảng song song 5.00s, 33444 kết quả


n = q = 2·10^4: chia khối 0.52s | treap 0.74s


**Ghi chú hiệu năng:** treap thuần Python ở giới hạn `n = q = 10^5` mất khoảng 5 giây (mỗi thao tác vài trăm lời gọi hàm đệ quy). Đây là giới hạn của Python; cùng thuật toán bằng C++/Java chạy dưới 0,2 giây.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên `push` trước khi đi xuống trong `split`

In [14]:
def split_no_push(t, k):
    if t is None:
        return None, None
    if size(t.left) >= k:                      # SAI: chưa đẩy cờ rev
        a, b = split_no_push(t.left, k); t.left = b; pull(t); return a, t
    a, b = split_no_push(t.right, k - size(t.left) - 1); t.right = a; pull(t); return t, b

def run_with_split(nums, queries, splitf, seed=3):
    rnd = random.Random(seed)
    root = None
    for x in nums:
        root = merge(root, Node(x, rnd.random()))
    res = []
    for t, x, y in queries:
        a, b = splitf(root, x)
        if t == 1:
            m, c = splitf(b, 1); m.val = m.agg = y
        else:
            m, c = splitf(b, y - x + 1)
            if t == 2: res.append(m.agg)
            else: m.rev = not m.rev
        root = merge(a, merge(m, c))
    return res

random.seed(11)
found = None
for _ in range(500):
    n = random.randint(2, 8)
    nums = [random.randint(0, 15) for _ in range(n)]
    qs = rand_queries(n, 10, 15)
    if run_with_split(nums, qs, split_no_push) != range_xor_brute(nums, qs):
        found = (nums, qs); break
print('phản ví dụ:', found)
print('sai :', run_with_split(*found, split_no_push), '| đúng:', range_xor_brute(*found))

phản ví dụ: ([14, 14, 6, 5, 15], [[3, 4, 4], [1, 3, 9], [1, 0, 1], [3, 3, 4], [3, 4, 4], [3, 0, 4], [1, 0, 1], [1, 1, 0], [2, 2, 3], [3, 1, 2]])
sai : [1] | đúng: [8]


**Vì sao sai:** nếu chưa áp dụng cờ đảo, "cây con trái" thực ra là phần **bên phải** của đoạn ⇒ tách sai vị trí.

### Lỗi 2 — Tưởng XOR không đổi nên bỏ qua hẳn truy vấn đảo

In [15]:
def ignore_reverse(nums, queries):
    return range_xor_brute(nums, [q for q in queries if q[0] != 3])

print('sai :', ignore_reverse(*examples[0][:2]), '| đúng:', examples[0][2])
case = ([1, 2, 4], [[3, 0, 1], [2, 0, 0]])
print('sai :', ignore_reverse(*case), '| đúng:', range_xor_brute(*case))

sai : [5, 8] | đúng: [5, 8]
sai : [1] | đúng: [2]


**Vì sao sai:** XOR của **đoạn bị đảo** không đổi, nhưng truy vấn sau có thể chỉ lấy **một phần** của đoạn đó (`[2,1,4]` → XOR `[0,0]` là 2, không phải 1).

### Lỗi 3 — Đổi chỗ con nhưng quên **lật** (toggle) cờ của con, gán `True`

In [16]:
def push_wrong(t):
    if t.rev:
        t.left, t.right = t.right, t.left
        if t.left: t.left.rev = True          # SAI: đảo hai lần phải trở về như cũ
        if t.right: t.right.rev = True
        t.rev = False

def make_ops(pushf):
    def sp(t, k):
        if t is None: return None, None
        pushf(t)
        if size(t.left) >= k:
            a, b = sp(t.left, k); t.left = b; pull(t); return a, t
        a, b = sp(t.right, k - size(t.left) - 1); t.right = a; pull(t); return t, b
    def mg(a, b):
        if a is None or b is None: return a or b
        if a.pri > b.pri:
            pushf(a); a.right = mg(a.right, b); pull(a); return a
        pushf(b); b.left = mg(a, b.left); pull(b); return b
    return sp, mg

def run_with_push(nums, queries, pushf, seed=3):
    sp, mg = make_ops(pushf)
    rnd = random.Random(seed)
    root = None
    for x in nums:
        root = mg(root, Node(x, rnd.random()))
    res = []
    for t, x, y in queries:
        a, b = sp(root, x)
        if t == 1:
            m, c = sp(b, 1); m.val = m.agg = y
        else:
            m, c = sp(b, y - x + 1)
            if t == 2: res.append(m.agg)
            else: m.rev = not m.rev
        root = mg(a, mg(m, c))
    return res

random.seed(12)
found = None
for _ in range(2000):
    n = random.randint(2, 8)
    nums = [random.randint(0, 15) for _ in range(n)]
    qs = rand_queries(n, 12, 15)
    if run_with_push(nums, qs, push_wrong) != range_xor_brute(nums, qs):
        found = (nums, qs); break
print('phản ví dụ:', found)
print('sai :', run_with_push(*found, push_wrong), '| đúng:', range_xor_brute(*found))

phản ví dụ: ([3, 4, 15, 15, 15, 3, 8], [[3, 1, 1], [3, 0, 1], [3, 0, 6], [3, 1, 6], [2, 3, 4], [3, 4, 5], [1, 5, 0], [3, 4, 4], [1, 6, 14], [2, 5, 6], [2, 0, 0], [1, 3, 1]])
sai : [12, 14, 8] | đúng: [0, 14, 8]


**Vì sao sai:** hai lần đảo triệt tiêu nhau; gán `True` làm mất thông tin "đã đảo chẵn lần".

### Lỗi 4 — Dựng treap bằng chèn với khóa là **giá trị** (BST thường) thay vì vị trí

In [17]:
print('BST theo giá trị sắp lại mảng:', sorted([3, 1, 2]), '— mất thứ tự gốc [3, 1, 2]; treap ẩn chỉ số dùng VỊ TRÍ (size) làm khóa ngầm.')

BST theo giá trị sắp lại mảng: [1, 2, 3] — mất thứ tự gốc [3, 1, 2]; treap ẩn chỉ số dùng VỊ TRÍ (size) làm khóa ngầm.


**Vì sao sai:** bài cần giữ **thứ tự vị trí**; khóa phải là chỉ số ngầm (suy ra từ `size`), không phải giá trị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đảo đoạn / cắt-dán đoạn / chèn-xóa giữa mảng" + truy vấn tổng hợp ⇒ **treap ẩn chỉ số** (hoặc splay tree, rope).
- Tổng hợp **giao hoán** (XOR, tổng, min, max) ⇒ đảo không đổi giá trị tổng hợp; tổng hợp **có hướng** (băm, chuỗi) ⇒ lưu cả hai chiều.
- Trong Python: mảng song song + dựng `O(n)` nhanh hơn đối tượng.
- Thay thế đơn giản hơn: chia khối `√n`.

## 15. Biến thể của bài

1. Thay XOR bằng **tổng** hoặc **min/max** ⇒ cùng cấu trúc.
2. Thêm truy vấn "**cộng `v` cho cả đoạn**" ⇒ thêm lazy cộng (với tổng: `agg += v·size`).
3. Hỏi **băm** của đoạn (so sánh chuỗi con) ⇒ lưu băm thuận và băm ngược, đảo thì đổi chỗ.
4. Bài liên quan: LC 1825, các bài "rope" / SPOJ "REVERSE", POJ 3580 *SuperMemo*.

In [18]:
# Biến thể 1: tổng đoạn với đảo (đổi phép XOR thành +)
def range_sum_brute(nums, queries):
    a, res = nums[:], []
    for t, x, y in queries:
        if t == 1: a[x] = y
        elif t == 2: res.append(sum(a[x:y + 1]))
        else: a[x:y + 1] = a[x:y + 1][::-1]
    return res

class SumNode(Node):
    pass

def range_sum_treap(nums, queries, seed=1):
    rnd = random.Random(seed)
    def pull_s(t):
        t.size = 1 + size(t.left) + size(t.right)
        t.agg = t.val + (t.left.agg if t.left else 0) + (t.right.agg if t.right else 0)
    def split_s(t, k):
        if t is None: return None, None
        push(t)
        if size(t.left) >= k:
            a, b = split_s(t.left, k); t.left = b; pull_s(t); return a, t
        a, b = split_s(t.right, k - size(t.left) - 1); t.right = a; pull_s(t); return t, b
    def merge_s(a, b):
        if a is None or b is None: return a or b
        if a.pri > b.pri:
            push(a); a.right = merge_s(a.right, b); pull_s(a); return a
        push(b); b.left = merge_s(a, b.left); pull_s(b); return b
    root = None
    for x in nums:
        root = merge_s(root, SumNode(x, rnd.random()))
    res = []
    for t, x, y in queries:
        a, b = split_s(root, x)
        if t == 1:
            m, c = split_s(b, 1); m.val = m.agg = y
        else:
            m, c = split_s(b, y - x + 1)
            if t == 2: res.append(m.agg)
            else: m.rev = not m.rev
        root = merge_s(a, merge_s(m, c))
    return res

random.seed(15)
for _ in range(200):
    n = random.randint(1, 20)
    nums = [random.randint(0, 50) for _ in range(n)]
    qs = rand_queries(n, 20, 50)
    assert range_sum_treap(nums, qs) == range_sum_brute(nums, qs)
print('Biến thể tổng đoạn: khớp mô phỏng.')

Biến thể tổng đoạn: khớp mô phỏng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Mỗi truy vấn | Khi nào dùng |
|---|---|---|---|
| 1 | Mô phỏng list | `O(n)` | Oracle / `n·q` nhỏ |
| 2 | Chia khối + cờ đảo | `O(√n)` | Khi ngại cài cây cân bằng |
| 3 | Treap đối tượng | `O(log n)` | Dễ đọc, dạy học |
| 4 | Treap mảng song song | `O(log n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 2, 3]`, truy vấn `[[3, 0, 2], [2, 0, 0], [2, 1, 2]]`.

**Bài 2 (tính tay):** `nums = [4, 4, 1]`, `[[1, 1, 6], [3, 1, 2], [2, 0, 1]]`.

**Bài 3 (điền TODO):** viết `kth(root, k)` trả về giá trị phần tử thứ `k` (0-based) của treap.

**Bài 4 (tự viết lại):** thêm truy vấn loại 4 `[4, l, r]` trả về **max** của đoạn (lưu thêm `mx` trong nút).

**Bài 5 (mini challenge):** thêm truy vấn "cắt đoạn `[l, r]` và dán vào đầu mảng" (cut-paste) và kiểm chứng bằng mô phỏng.

In [19]:
def kth_todo(root, k):
    # TODO: push, so sánh k với size(left)
    raise NotImplementedError

def with_max_todo(nums, queries):
    # TODO
    raise NotImplementedError

def cut_paste_todo(nums, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Sau đảo: `[3, 2, 1]`. (2) XOR `[0,0] = 3`, XOR `[1,2] = 2^1 = 3`. (3) `[3, 3]`.

**Bài 2.** (1) `[4, 6, 1]`. (2) Đảo `[1,2]`: `[4, 1, 6]`. (3) XOR `[0,1] = 4^1 = 5`.

**Bài 3.** (1) `push(t)`. (2) Nếu `k < size(left)` đi trái; bằng thì trả `val`; lớn hơn đi phải với `k − size(left) − 1`. (3) So với `to_list`.

**Bài 4.** (1) `mx = max(val, mx(trái), mx(phải))`. (2) Max cũng giao hoán ⇒ đảo không đổi. (3) Vét cạn kiểm tra.

**Bài 5.** (1) `split` thành `A, M, C`. (2) Ghép `M, A, C`. (3) Mô phỏng bằng slicing.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().getResults([1, 2, 3], [[3, 0, 2], [2, 0, 0], [2, 1, 2]])
a2 = Solution().getResults([4, 4, 1], [[1, 1, 6], [3, 1, 2], [2, 0, 1]])
assert a1 == [3, 3] and a2 == [5]

def kth(t, k):
    while t:
        push(t)
        ls = size(t.left)
        if k < ls:
            t = t.left
        elif k == ls:
            return t.val
        else:
            k -= ls + 1
            t = t.right

rnd = random.Random(4)
root = None
for x in [9, 8, 7, 6, 5]:
    root = merge(root, Node(x, rnd.random()))
a, b = split(root, 1); m, c = split(b, 3); m.rev = True; root = merge(a, merge(m, c))
assert [kth(root, i) for i in range(5)] == to_list(root) == [9, 6, 7, 8, 5]
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3:', [kth(root, i) for i in range(5)])

Bài 1: [3, 3] | Bài 2: [5] | Bài 3: [9, 6, 7, 8, 5]


In [21]:
# Bài 4 và 5 (dùng cấu trúc đối tượng, thêm trường mx bằng thuộc tính ngoài __slots__ qua lớp con)
class MaxNode:
    __slots__ = ('val', 'agg', 'mx', 'size', 'pri', 'left', 'right', 'rev')
    def __init__(self, val, pri):
        self.val = self.agg = self.mx = val
        self.size, self.pri, self.left, self.right, self.rev = 1, pri, None, None, False

def pull_m(t):
    t.size = 1 + size(t.left) + size(t.right)
    t.agg = t.val ^ agg(t.left) ^ agg(t.right)
    t.mx = max(t.val, t.left.mx if t.left else -1, t.right.mx if t.right else -1)

def split_m(t, k):
    if t is None: return None, None
    push(t)
    if size(t.left) >= k:
        a, b = split_m(t.left, k); t.left = b; pull_m(t); return a, t
    a, b = split_m(t.right, k - size(t.left) - 1); t.right = a; pull_m(t); return t, b

def merge_m(a, b):
    if a is None or b is None: return a or b
    if a.pri > b.pri:
        push(a); a.right = merge_m(a.right, b); pull_m(a); return a
    push(b); b.left = merge_m(a, b.left); pull_m(b); return b

def extended(nums, queries, seed=5):
    rnd = random.Random(seed)
    root = None
    for x in nums:
        root = merge_m(root, MaxNode(x, rnd.random()))
    res = []
    for t, x, y in queries:
        a, b = split_m(root, x)
        if t == 1:
            m, c = split_m(b, 1); m.val = m.agg = m.mx = y; root = merge_m(a, merge_m(m, c)); continue
        m, c = split_m(b, y - x + 1)
        if t == 2: res.append(m.agg)
        elif t == 3: m.rev = not m.rev
        elif t == 4: res.append(m.mx)
        if t == 5:
            root = merge_m(m, merge_m(a, c))          # cắt đoạn dán lên đầu
        else:
            root = merge_m(a, merge_m(m, c))
    return res

def extended_brute(nums, queries):
    a, res = nums[:], []
    for t, x, y in queries:
        if t == 1: a[x] = y
        elif t == 2: res.append(reduce(xor, a[x:y + 1], 0))
        elif t == 3: a[x:y + 1] = a[x:y + 1][::-1]
        elif t == 4: res.append(max(a[x:y + 1]))
        else: a = a[x:y + 1] + a[:x] + a[y + 1:]
    return res

random.seed(16)
for _ in range(300):
    n = random.randint(1, 15)
    nums = [random.randint(0, 40) for _ in range(n)]
    qs = []
    for _ in range(15):
        t = random.randint(1, 5)
        if t == 1: qs.append([1, random.randrange(n), random.randint(0, 40)])
        else:
            l = random.randrange(n); qs.append([t, l, random.randrange(l, n)])
    assert extended(nums, qs) == extended_brute(nums, qs)
print('Bài 4 & 5: max đoạn và cắt-dán khớp mô phỏng.')

Bài 4 & 5: max đoạn và cắt-dán khớp mô phỏng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nút: val, agg(XOR cây con), size, pri ngẫu nhiên, left, right, rev
push: nếu rev -> đổi trái/phải, lật rev của con
split(t, k): push; nếu size(left) >= k đi trái, ngược lại đi phải với k - size(left) - 1; pull
merge(a, b): pri lớn làm gốc; push; ghép đệ quy; pull
truy vấn [l, r]: A, B = split(root, l); M, C = split(B, r-l+1); ...; root = merge(A, merge(M, C))
```

**Checklist:**

- [ ] `push` trước khi đi xuống trong cả `split` và `merge`?
- [ ] Cờ đảo dùng phép **lật**, không gán `True`?
- [ ] `pull` sau mọi thay đổi con?
- [ ] Gán điểm: cập nhật cả `val` và `agg` của nút lá đã tách?
- [ ] Đối chiếu với mô phỏng; chú ý tên hàm suy luận (`getResults`).